In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
!pip install micromlgen
import joblib

# Ensure micromlgen is installed before running: pip install micromlgen
try:
    from micromlgen import port
except ImportError:
    print("micromlgen not found. Please run: pip install micromlgen")

# ==========================================
# 1. Configuration & Setup
# ==========================================
PERSON_FOLDERS = [
    "/content/drive/MyDrive/dataset/Person1",
    "/content/drive/MyDrive/dataset/Person2",
    "/content/drive/MyDrive/dataset/Person3"
]
MODEL_EXPORT_PATH = 'mmwave_rf4_model.pkl'
C_MODEL_EXPORT_PATH = 'mmwave_rf4_model.h' # Export path for the ESP32 Microcontroller

# Target Features for Quartile Plots
PLOT_FEATURES = {
    'Distance (mm)': 'rd_distance_mm',
    'Velocity (cm/s)': 'rd_velocity_cm_s',
    'Angle (deg)': 'rd_angle_deg'
}

# ==========================================
# Mock Data Generator (For out-of-the-box testing)
# ==========================================
def generate_mock_data():
    """Generates mock data if the user's dataset directory is not found."""
    print("Real dataset not found. Generating mock data for demonstration...")
    classes = ['Standing', 'Sitting', 'Entry_Exit', 'Standing_to_Sitting', 'Sitting_to_Standing', 'Walking', 'No_Person']

    for person_dir in PERSON_FOLDERS:
        os.makedirs(person_dir, exist_ok=True)
        person_name = os.path.basename(person_dir)
        for cls in classes:
            # Generate random baseline data
            rows = 100 if cls != 'Entry_Exit' else 200
            data = {
                'datetime': pd.date_range(start='2026-05-01', periods=rows, freq='S'),
                'wifi': [1]*rows, 'time_sync': [1]*rows, 'rd_present': [1]*rows,
                'rd_distance_mm': np.random.normal(1000 if cls != 'Walking' else 2000, 200, rows),
                'rd_velocity_cm_s': np.random.normal(0, 5, rows),
                'rd_angle_deg': np.random.normal(10, 5, rows),
                'rd_x_mm': np.random.normal(500, 100, rows),
                'rd_y_mm': np.random.normal(500, 100, rows),
                'c4001_present': [1]*rows,
                'c4001_distance_m': np.random.normal(1.0, 0.2, rows),
                'c4001_velocity_m_s': np.random.normal(0, 0.05, rows),
                'c4001_energy': np.random.normal(50000, 5000, rows),
                'c4001_raw_targets': [1]*rows,
                'c4001_raw_distance_m': np.random.normal(1.0, 0.2, rows),
                'c4001_raw_velocity_m_s': np.random.normal(0, 0.05, rows),
                'c4001_raw_energy': np.random.normal(50000, 5000, rows),
            }

            # Apply class specific rules
            if cls == 'No_Person':
                for key in data:
                    if 'present' in key or 'distance' in key or 'velocity' in key or 'energy' in key:
                        data[key] = [0]*rows
            elif cls == 'Entry_Exit':
                # Half entry (negative velocity), half exit (positive velocity)
                data['rd_velocity_cm_s'] = np.concatenate([np.random.normal(-50, 10, 100), np.random.normal(50, 10, 100)])

            df = pd.DataFrame(data)
            df.to_csv(os.path.join(person_dir, f'{cls}_{person_name}.csv'), index=False)

# ==========================================
# 2. Data Labeling & Preprocessing
# ==========================================
def load_and_label_data():
    all_data = []

    # Check if any of the specific folders exist
    if not any(os.path.exists(folder) for folder in PERSON_FOLDERS):
        generate_mock_data()

    for person_dir in PERSON_FOLDERS:
        if not os.path.exists(person_dir):
            print(f"Warning: Folder not found -> {person_dir}")
            continue

        person_name = os.path.basename(person_dir)
        csv_files = glob.glob(os.path.join(person_dir, "*.csv"))

        for file in csv_files:
            filename = os.path.basename(file).lower()
            df = pd.read_csv(file)

            # Determine base class from filename
            if 'entry_exit' in filename:
                # Apply Thresholding Logic for Entry vs Exit based on velocity
                # Entry means velocity < 0, Exit means velocity >= 0
                df['Activity'] = df.apply(
                    lambda row: 'Entry' if row['rd_velocity_cm_s'] < 0 else 'Exit',
                    axis=1
                )
            elif 'no_person' in filename:
                # Ensure No Person has 0 values (Cleaning step to enforce rule)
                cols_to_zero = [c for c in df.columns if c not in ['datetime', 'wifi', 'time_sync']]
                df[cols_to_zero] = 0
                df['Activity'] = 'No_Person'
            elif 'standing_to_sitting' in filename: df['Activity'] = 'Standing_to_Sitting'
            elif 'sitting_to_standing' in filename: df['Activity'] = 'Sitting_to_Standing'
            elif 'standing' in filename: df['Activity'] = 'Standing'
            elif 'sitting' in filename: df['Activity'] = 'Sitting'
            elif 'walking' in filename: df['Activity'] = 'Walking'
            else: continue # Skip unknown files

            df['Person'] = person_name
            all_data.append(df)

    if not all_data:
        raise ValueError("No valid CSV files found to process.")

    combined_df = pd.concat(all_data, ignore_index=True)

    # Drop non-predictive features
    cols_to_drop = ['datetime', 'wifi', 'time_sync']
    combined_df = combined_df.drop(columns=[col for col in cols_to_drop if col in combined_df.columns])

    # Fill any missing values with 0
    combined_df = combined_df.fillna(0)

    return combined_df

# ==========================================
# 3. Noise Reduction (EMA Only)
# ==========================================
def reduce_noise(df, features, ema_span=5):
    """
    Reduces mmWave multipath noise by applying an
    Exponential Moving Average (EMA) to smooth out high-frequency jitter.

    Args:
        df: The dataframe containing the sensor data.
        features: List of feature column names to process.
        ema_span: The span for the EMA window (default 5). Higher = smoother but more lag.
    """
    df_clean = df.copy()

    for col in features:
        if df_clean[col].dtype in ['float64', 'int64']:
            # Exponential Moving Average (Smooths out jitter)
            # adjust=False calculates it recursively, mirroring C++ implementation
            df_clean[col] = df_clean[col].ewm(span=ema_span, adjust=False).mean()

    return df_clean

# ==========================================
# 4. EDA & Visualizations (Quartile Plots)
# ==========================================
def generate_quartile_plots(df):
    sns.set_theme(style="whitegrid")

    # 4a. Quartile Plots for Each Person
    for person in df['Person'].unique():
        person_df = df[df['Person'] == person]
        fig, axes = plt.subplots(3, 1, figsize=(12, 15))
        fig.suptitle(f'Sensor Feature Quartiles by Activity - {person}', fontsize=16, y=0.98)

        for i, (title, feature) in enumerate(PLOT_FEATURES.items()):
            sns.boxplot(ax=axes[i], data=person_df, x='Activity', y=feature, palette='Set2')
            axes[i].set_title(f'{title} Distribution')
            axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=15)

        plt.tight_layout()
        plt.savefig(f'{person}_quartile_plots.png')
        plt.close()
        print(f"Saved {person}_quartile_plots.png")

    # 4b. Merged Quartile Plot for Comparison
    fig, axes = plt.subplots(3, 1, figsize=(15, 18))
    fig.suptitle('Merged Feature Quartiles Comparison (All Persons)', fontsize=18, y=0.98)

    for i, (title, feature) in enumerate(PLOT_FEATURES.items()):
        sns.boxplot(ax=axes[i], data=df, x='Activity', y=feature, hue='Person', palette='muted')
        axes[i].set_title(f'{title} Comparison Across Persons')
        axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=15)
        axes[i].legend(loc='upper right')

    plt.tight_layout()
    plt.savefig('Merged_Persons_Quartile_Plots.png')
    plt.close()
    print("Saved Merged_Persons_Quartile_Plots.png")

# ==========================================
# 5. ML Pipeline Execution
# ==========================================
def main():
    print("Starting ML Pipeline...")

    # Load and Label
    print("1. Loading and Labeling Data...")
    df = load_and_label_data()

    # Generate EDA Visualizations before scaling/balancing
    print("2. Generating Quartile Plots...")
    generate_quartile_plots(df)

    # Prepare X and y
    feature_columns = [col for col in df.columns if col not in ['Activity', 'Person']]
    X = df[feature_columns]
    y = df['Activity']

    # Noise Reduction
    print("3. Applying Noise Reduction (EMA Smoothing)...")
    X = reduce_noise(X, feature_columns, ema_span=5)

    # Generate Feature Correlation Heatmap
    print("4. Generating Correlation Heatmap...")
    plt.figure(figsize=(12, 10))
    corr_matrix = X.corr()
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True, square=True)
    plt.title('Feature Correlation Heatmap')
    plt.tight_layout()
    plt.savefig('Feature_Correlation_Heatmap.png')
    plt.close()
    print("Saved Feature_Correlation_Heatmap.png")

    # Encoding Labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # Train-Test Split (Stratified to maintain class distributions)
    print("5. Splitting Data (80% Train, 20% Test)...")
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

    # Feature Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- BEFORE SMOTE METRICS ---
    print("\n" + "-"*40)
    print("--- Class Distribution BEFORE SMOTE ---")
    print("-"*40)
    classes_before, counts_before = np.unique(y_train, return_counts=True)
    for cls, count in zip(classes_before, counts_before):
        print(f"{label_encoder.inverse_transform([cls])[0]:<20} : {count}")
    print(f"\nTotal Samples BEFORE SMOTE: {len(y_train)}\n")

    # SMOTE (Data Balancing on Training set ONLY)
    print("6. Applying SMOTE to balance classes...")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

    # --- AFTER SMOTE METRICS ---
    print("\n" + "-"*40)
    print("--- Class Distribution AFTER SMOTE ---")
    print("-"*40)
    classes_after, counts_after = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(classes_after, counts_after):
        print(f"{label_encoder.inverse_transform([cls])[0]:<20} : {count}")
    print(f"\nTotal Samples AFTER SMOTE: {len(y_train_resampled)}\n")

    # Model Training & Optimization
    print("7. Training Random Forest Classifier (Optimized for minimal memory footprint)...")
    clf = RandomForestClassifier(
        n_estimators=25,            # Reduced to 25: Drastically shrinks the exported C code size
        max_depth=8,                # Depth 8: Prevents exponential tree growth, minimizing flash/RAM usage
        min_samples_split=5,        # Increased to 5: Prunes highly specific micro-branches
        min_samples_leaf=2,         # Smooths out noise
        class_weight='balanced',    # Helps with transient activities
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train_resampled, y_train_resampled)

    # Model Evaluation
    print("8. Evaluating Model...")
    y_pred = clf.predict(X_test_scaled)

    overall_accuracy = accuracy_score(y_test, y_pred)
    print("\n" + "="*40)
    print(f"OVERALL MODEL ACCURACY: {overall_accuracy * 100:.2f}%")
    print("="*40 + "\n")

    print("Classification Report:")
    target_names = label_encoder.inverse_transform(np.unique(y_test))
    print(classification_report(y_test, y_pred, target_names=target_names))

    # Confusion Matrix (0, 1 Format / Normalized)
    print("9. Generating Normalized Confusion Matrix...")
    cm_normalized = confusion_matrix(y_test, y_pred, normalize='true')

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.ylabel('Actual Activity')
    plt.xlabel('Predicted Activity')
    plt.title('Normalized Confusion Matrix (0 to 1 Format)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('Normalized_Confusion_Matrix.png')
    plt.close()
    print("Saved Normalized_Confusion_Matrix.png")

    # Model Export (PC Version)
    print("10. Exporting Pipeline (joblib)...")
    pipeline_export = {
        'model': clf,
        'scaler': scaler,
        'label_encoder': label_encoder,
        'features': feature_columns
    }
    joblib.dump(pipeline_export, MODEL_EXPORT_PATH)
    print(f"Pipeline successfully exported to {MODEL_EXPORT_PATH}")

    # Model Export (ESP32 Microcontroller Version)
    print("11. Exporting C++ model for ESP32 using micromlgen...")
    try:
        # Map labels to actual class names so the ESP32 code outputs strings instead of integers
        classmap = {i: str(cls) for i, cls in enumerate(label_encoder.classes_)}
        c_code = port(clf, classmap=classmap)

        # Append Scaler parameters to the C header for ESP32 preprocessing
        feature_names_c = '"' + '", "'.join(feature_columns) + '"'
        scaler_code = f"""
// ==========================================
// StandardScaler Parameters for ESP32
// Note: Apply these before passing features to predict()
// Example: feature_scaled[i] = (feature_raw[i] - scaler_means[i]) / scaler_scales[i];
// ==========================================
const float scaler_means[] = {{{', '.join(map(str, scaler.mean_))}}};
const float scaler_scales[] = {{{', '.join(map(str, scaler.scale_))}}};
const char* feature_names[] = {{{feature_names_c}}};
"""
        with open(C_MODEL_EXPORT_PATH, 'w') as f:
            f.write(c_code)
            f.write("\n" + scaler_code)
        print(f"C model and scaler params successfully exported to {C_MODEL_EXPORT_PATH} for ESP32-S3 Pico.")
    except Exception as e:
        print(f"Failed to export C model: {e}")
        print("Tip: Ensure you have installed micromlgen ('pip install micromlgen').")

    print("Done!")

if __name__ == "__main__":
    main()

  Preparing metadata (setup.py) ... done
  Created wheel for micromlgen: filename=micromlgen-1.1.28-py3-none-any.whl size=32152 sha256=1371fd31422205b73ce3b3baaabf5f991c372b26a3afc223435d3c8a6133f8a9
  Stored in directory: /root/.cache/pip/wheels/16/02/8a/3a8b533174e4f7691a8fd72dab4493fb6819b79f8fcc1d18a6
Successfully built micromlgen
Starting ML Pipeline...
1. Loading and Labeling Data...
2. Generating Quartile Plots...


/tmp/ipykernel_1117/2269710467.py:178: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(ax=axes[i], data=person_df, x='Activity', y=feature, palette='Set2')
/tmp/ipykernel_1117/2269710467.py:180: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=15)
/tmp/ipykernel_1117/2269710467.py:178: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(ax=axes[i], data=person_df, x='Activity', y=feature, palette='Set2')
/tmp/ipykernel_1117/2269710467.py:180: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a Fixe

Saved Person1_quartile_plots.png


/tmp/ipykernel_1117/2269710467.py:178: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(ax=axes[i], data=person_df, x='Activity', y=feature, palette='Set2')
/tmp/ipykernel_1117/2269710467.py:180: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=15)
/tmp/ipykernel_1117/2269710467.py:178: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(ax=axes[i], data=person_df, x='Activity', y=feature, palette='Set2')
/tmp/ipykernel_1117/2269710467.py:180: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a Fixe

Saved Person2_quartile_plots.png


/tmp/ipykernel_1117/2269710467.py:178: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(ax=axes[i], data=person_df, x='Activity', y=feature, palette='Set2')
/tmp/ipykernel_1117/2269710467.py:180: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=15)
/tmp/ipykernel_1117/2269710467.py:178: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(ax=axes[i], data=person_df, x='Activity', y=feature, palette='Set2')
/tmp/ipykernel_1117/2269710467.py:180: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a Fixe

Saved Person3_quartile_plots.png


/tmp/ipykernel_1117/2269710467.py:194: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=15)
/tmp/ipykernel_1117/2269710467.py:194: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=15)
/tmp/ipykernel_1117/2269710467.py:194: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=15)


Saved Merged_Persons_Quartile_Plots.png
3. Applying Noise Reduction (EMA Smoothing)...
4. Generating Correlation Heatmap...
Saved Feature_Correlation_Heatmap.png
5. Splitting Data (80% Train, 20% Test)...

----------------------------------------
--- Class Distribution BEFORE SMOTE ---
----------------------------------------
Entry                : 12578
Exit                 : 14826
No_Person            : 12000
Sitting              : 15357
Sitting_to_Standing  : 23340
Standing             : 13918
Standing_to_Sitting  : 20689
Walking              : 14207

Total Samples BEFORE SMOTE: 126915

6. Applying SMOTE to balance classes...

----------------------------------------
--- Class Distribution AFTER SMOTE ---
----------------------------------------
Entry                : 23340
Exit                 : 23340
No_Person            : 23340
Sitting              : 23340
Sitting_to_Standing  : 23340
Standing             : 23340
Standing_to_Sitting  : 23340
Walking              : 23340

Total Sa